# Week 9 part 2: Linear regression examples

In Part 1 you saw how to use the SVD to compute OLS (ordinary least squares) regression. You also saw how to compute linear regression using scikit-learn. In this segment you'll use linear regression to study salary data.

## Initialization cells

In [ ]:
import numpy as np
import numpy.linalg as la

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors

#import plotly.graph_objs as go
#import plotly.express as px

import pandas as pd

rng = np.random.default_rng()

## End of initialization cells

## US population 

Start by reading in some us population information as a dataframe

In [ ]:
uspop_data = pd.read_csv("USPop.csv", dtype={'year': np.int64, 'population': np.double})
uspop_data.head() # print out first 5 rows of the data

In [ ]:
uspop_data.tail() # print out the last five rows of data

This data set contains the list of US populations from 1900 to 2020 (I obtained these from Wolfram Alpha).

Let's use linear regression to model the population during the "baby boom" from roughly 1946 to 1964.

In [ ]:
# This puts the population from 1946 to 1964 in an array

PopBoom = uspop_data['population'].values[46:64]
YearsBoom = uspop_data['year'].values[46:64]

# the numpy function np.round rounds to the specified number of decimal places.

#np.round(PopBoom[1:]/PopBoom[0:-1],decimals=3)
pop = PopBoom[:5]
yrs = YearsBoom[:5]

print(pop)
print(yrs)

In [ ]:
# Now let's model the data 

pop = PopBoom[:5]
yrs = YearsBoom[:5]

A = np.array([yrs,np.ones_like(yrs)]).T

H,s,CT = la.svd(A,full_matrices=False)

# This is our formula for the slope and intercept using the SVD of A

r,b = (CT.T * (1/s)) @ H.T @ pop

# Let's plot the population along with the 
# estimated population 

fig,ax = plt.subplots()

ax.scatter(YearsBoom,PopBoom)
ax.plot(YearsBoom,r*YearsBoom + b,color='orange')

In [ ]:
def R_squared(Y,Y_est):
    SSresiduals = (Y-Y_est)@(Y-Y_est)
    SSY = (Y-Y.mean()) @ (Y-Y.mean())
    return 1. - SSresiduals/SSY
R_squared (PopBoom,r*YearsBoom + b)

Even over the sample where our estimate is visually failing, $R^2$ tells us we're capturing a lot of the variance! But let's look at the average error.

In [ ]:
Residuals = (PopBoom -  (r*YearsBoom +b))
print(f"{Residuals[:5].mean():0.2f}")
print(f"{Residuals.mean():0.2f}")


Over the period we estimated, linear regression forces the mean error to be zero. But over the longer 20 year period, we can see that our estimate is consistently an understimate. The failure average error to be zero is called "bias", and it is one sign that your model may need fixing.

In this case, the problem is that during this period the population is growing closer to exponentially. Let's print $p_t/p_{t-1}$ for the period we're studying.

In [ ]:
print(np.around(PopBoom[1:]/PopBoom[:-1],decimals=3))

The ratio $p_t/p_{t-1}$ is noearly constant: this is *not* a feature of linear functions of time. So the model

$$
   p = m t + b
$$

isn't a plausible model for this data. But a constaint ratio $p_t/p_{t-1}$ *is* a feature of exponential growth. If

$$
  p_t =  a e^{rt},
$$

then the ratio is constant:

$$
 \frac{p_{t+1}}{p_t} = e^r!
$$

The nearly constant ratio suggests modeling the population as 

$$
   p_t = a e^{rt}
$$

or equivalently

$$
   \log p_t = r t + b:
$$

in this model the logarithm of the population is a linear function of time. So instead of modeling the population using linear regression, we'll model the log of the population using linear regression.

In [ ]:
# It's easy to add another column to a data frame...

uspop_data['logpop'] = np.log(uspop_data['population'])
uspop_data.head()

In [ ]:
# Now let's model the data 

YearsBoom = uspop_data['year'].values[46:63]
LogPopBoom = uspop_data['logpop'].values[46:63]


A = np.array([YearsBoom[0:5],np.ones_like(YearsBoom[0:5])]).T

H,s,CT = la.svd(A,full_matrices=False)

# This is our formula for the slope and intercept using the SVD of A

r,b = (CT.T * (1/s)) @ H.T @ LogPopBoom[0:5]

# Let's the log of the population along with the 
# actual values over this period

fig,ax = plt.subplots()

ax.scatter(YearsBoom,LogPopBoom)
ax.plot(YearsBoom,r*YearsBoom+ b,color='orange')

print(R_squared (LogPopBoom,r*YearsBoom + b))
Residuals = (LogPopBoom -  (r*YearsBoom +b))
print(f"{Residuals[:5].mean():0.2f}")
print(f"{Residuals.mean():0.2f}")



Visually this is an excellent fit, but it only works because the rate of growth of population was relatively constant during this period. If you look at the log of the population over the whole time period of our data set, you'll see several times when the slope changed.


In [ ]:


fig,ax = plt.subplots(figsize=(12,8))
ax.plot(uspop_data['year'].values,uspop_data['logpop'])




## Modeling salary

In [ ]:
salary_data = pd.read_csv("Salary_Data_Safe.csv")
salary_data.head()

In [ ]:
salary_data['LogSalary'] = np.log(salary_data['Salary'])
salary_data.tail()

In [ ]:
n = salary_data.shape[0]
A = np.array([salary_data['YearsSincePHD'],salary_data['Assoc'],salary_data['Full'],salary_data['CalledM'],np.ones(n)]).T

In [ ]:
salary_data.shape

In [ ]:
n = salary_data.shape[0]
A = np.array([salary_data['YearsSincePHD'],salary_data['Assoc'],salary_data['Full'],salary_data['CalledM'],np.ones(n)]).T

In [ ]:
A

In [ ]:

H,s,CT = la.svd(A,full_matrices=False)
m = (CT.T * (1/s)) @ H.T @ salary_data['Salary']

In [ ]:
# The RHS variables are 
#([salary_data['YearsSincePHD'],salary_data['Assoc'],salary_data['Full'],salary_data['CalledM'],np.ones(n)]).T
m

In [ ]:
fig,ax = plt.subplots(figsize=(10,10))

predicted_salaries = A @m 
ax.scatter(predicted_salaries,salary_data['Salary'])
slope_one = np.linspace(75000,225000)
ax.plot(slope_one,slope_one)

In [ ]:
salary_data.head()

In [ ]:
import statsmodels.api as sm
from patsy import dmatrices

In [ ]:
y, X = dmatrices('Salary ~ YearsSincePHD + Tenure_Rank + UIUC_Gender + DEPARTMENT', data=salary_data, return_type='dataframe')

In [ ]:
y[:10]

In [ ]:
X[:10]

In [ ]:
 mod = sm.OLS(y, X)
res=mod.fit()
print(res.summary())

In [ ]:
y, X = dmatrices('LogSalary ~ YearsSincePHD + Tenure_Rank + UIUC_Gender', data=salary_data, return_type='dataframe')

In [ ]:
 mod = sm.OLS(y, X) 
 res = mod.fit() 
print(res.summary())


In [ ]:
 res.params


In [ ]:
param_array = res.params.values
param_array

In [ ]:
param_array[np.array([4,1,2,3,0])]

In [ ]:
#A = np.array([salary_data['YearsSincePHD'],salary_data['Assoc'],salary_data['Full'],salary_data['CalledM'],np.ones(n)]
m = res.params.values[np.array([4,1,2,3,0])]


afig,ax = plt.subplots(figsize=(10,10))

predicted_log_salaries = A @m 
ax.scatter(predicted_log_salaries,salary_data['LogSalary'])
slope_one = np.linspace(11.2,12.4)
ax.plot(slope_one,slope_one)